# Kompresibilna sapnica: predvidi → izračunaj → provjeri

Za konvergentnu sapnicu s idealnim plinom snižavanje protutlaka najprije povećava maseni protok. Kada se u grlu dosegne \(M=1\), daljnje snižavanje protutlaka ne povećava protok kroz ovaj model.

## Predvidi

1. Skiciraj \(\dot m(p_b/p_0)\) i označi gdje očekuješ plato.
2. Ako se površina grla udvostruči, što se događa s prigušenim protokom?
3. Hoće li povećanje stagnacijske temperature povećati ili smanjiti \(\dot m_{max}\)?

Pretpostavke su kvazijednodimenzijski, izentropski idealni tok do grla i poznat koeficijent istjecanja. Model ne opisuje udarne valove ni tok nizvodno od grla.

Osnovna karakteristika u nastavku koristi ilustrativni $C_d=0,97$ i površinu
$50\ \mathrm{mm}^2$. Radne točke Z4 i neovisna procjena koeficijenta Z5
računaju se zasebno, s podatcima tih zadataka.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.dpi": 110, "font.size": 10})
gamma, R = 1.4, 287.0

def critical_pressure_ratio(gamma=gamma):
    return (2/(gamma+1))**(gamma/(gamma-1))

def choked_mass_flow(p0, T0, area, Cd=1.0, gamma=gamma, R=R):
    factor = np.sqrt(gamma/R)*(2/(gamma+1))**((gamma+1)/(2*(gamma-1)))
    return Cd*area*p0/np.sqrt(T0)*factor

def mass_flow(back_pressure_ratio, p0, T0, area, Cd=1.0):
    ratio = np.asarray(back_pressure_ratio, dtype=float)
    if np.any((ratio <= 0) | (ratio > 1)):
        raise ValueError("Treba vrijediti 0 < pb/p0 ≤ 1.")
    rcrit = critical_pressure_ratio()
    effective_ratio = np.maximum(ratio, rcrit)
    mach = np.sqrt(2/(gamma-1)*(effective_ratio**(-(gamma-1)/gamma)-1))
    flow_parameter = mach*(1+(gamma-1)*mach**2/2)**(-(gamma+1)/(2*(gamma-1)))
    dimensional = Cd*area*p0/np.sqrt(T0)*np.sqrt(gamma/R)
    return dimensional*flow_parameter

base = dict(p0=600e3, T0=300.0, area=50e-6, Cd=0.97)
rcrit = critical_pressure_ratio()
mdot_star = choked_mass_flow(**base)
print(f"Kritični omjer pb/p0 = {rcrit:.6f}")
print(f"Prigušeni maseni protok = {mdot_star:.6f} kg/s")


## Izračunaj: karakteristika i nesigurnost prigušenog protoka

Za male, neovisne ulazne nesigurnosti vrijedi približna relativna bilanca

\[
\left(\frac{u_{\dot m}}{\dot m}\right)^2=
\left(\frac{u_{p_0}}{p_0}\right)^2+
\left(\frac{u_A}{A}\right)^2+
\left(\frac{u_{C_d}}{C_d}\right)^2+
\frac14\left(\frac{u_{T_0}}{T_0}\right)^2.
\]

Provjeravamo je Monte Carlo uzorkovanjem s fiksnim sjemenom generatora.

Ove su ulazne standardne nesigurnosti posebno odabrane za numerički pokus;
nisu dodatni podatci zadatka Z5, u kojem su nesigurnosti zanemarene.


In [ ]:
ratios = np.linspace(0.08, 1.0, 300)
mdot_curve = mass_flow(ratios, **base)

sigma = dict(p0=3e3, T0=1.5, area=0.30e-6, Cd=0.005)
relative_linear = np.sqrt(
    (sigma["p0"]/base["p0"])**2 +
    (sigma["area"]/base["area"])**2 +
    (sigma["Cd"]/base["Cd"])**2 +
    0.25*(sigma["T0"]/base["T0"])**2
)
u_linear = mdot_star*relative_linear

rng = np.random.default_rng(20260802)
n_samples = 40_000
mc = {key: rng.normal(base[key], sigma[key], n_samples) for key in base}
mdot_mc = choked_mass_flow(**mc)
u_mc = np.std(mdot_mc, ddof=1)
interval = np.quantile(mdot_mc, [0.025, 0.975])
print(f"u_linear = {u_linear:.6f} kg/s; u_MC = {u_mc:.6f} kg/s")
print(f"95 %-tni Monte Carlo interval = [{interval[0]:.6f}, {interval[1]:.6f}] kg/s")


## Provjeri

Plato se provjerava na više protutlakova, geometrijsko skaliranje dvostrukom površinom, a linearna propagacija neovisnim Monte Carlo postupkom.

Nestlačiva usporedba zadržava gustoću komore $\rho_0=p_0/(RT_0)$ i računa
$\dot m=C_d A\sqrt{2\rho_0(p_0-p_b)}$. To je približna karakteristika za
malen pad tlaka, a ne model čitavoga raspona. Provjeravamo da pogreška
opada kada $p_b/p_0\to1$ i $Ma\to0$.


In [ ]:
plateau = mass_flow(np.array([0.10, 0.25, 0.50]), **base)
double_area = choked_mass_flow(**{**base, "area": 2*base["area"]})
near_one = float(mass_flow(np.array([1.0]), **base)[0])

assert np.allclose(plateau, mdot_star, rtol=1e-12)
assert np.isclose(double_area, 2*mdot_star, rtol=1e-13)
assert np.isclose(near_one, 0.0, atol=1e-14)
assert abs(u_mc/u_linear-1) < 0.05

rho0 = base["p0"]/(R*base["T0"])
def incompressible_mass_flow(ratio):
    return base["Cd"]*base["area"]*np.sqrt(2*rho0*base["p0"]*(1-ratio))

limit_ratios = np.array([0.99, 0.999, 0.9999])
isentropic_limit = mass_flow(limit_ratios, **base)
relative_error = incompressible_mass_flow(limit_ratios)/isentropic_limit - 1
assert np.all(np.diff(relative_error) < 0)
assert 0 < relative_error[-1] < 1e-4
print("Relativna pogreška nestlačive procjene pri približavanju pb/p0=1:", relative_error)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(ratios, mdot_curve, color="#256d85", lw=2)
axes[0].plot(ratios, incompressible_mass_flow(ratios), color="#3a4a56", ls="--", label="nestlačivo: gustoća komore")
axes[0].axvline(rcrit, color="#b43c35", ls="--", label="M=1 u grlu")
axes[0].set(xlabel="$p_b/p_0$", ylabel=r"$\dot m$ (kg/s)", title="Prigušenje masenog protoka")
axes[0].legend()
axes[1].hist(mdot_mc, bins=55, color="#7cb5d6", edgecolor="white")
axes[1].axvline(mdot_star, color="#b43c35", lw=2, label="nominalno")
axes[1].set(xlabel=r"$\dot m_{max}$ (kg/s)", ylabel="broj uzoraka", title="Ulazna nesigurnost")
axes[1].legend()
for ax in axes: ax.grid(True, ls=":", alpha=.45)
plt.tight_layout(); plt.show()


## Radne točke Z4 i provjera objašnjenja Z5

Prvo pretvori sve manometarske tlakove Z4 u apsolutne. Predvidi u kojem se
slučaju izlazni tlak smije poistovjetiti s protutlakom. Nakon toga za Z5
predvidi je li izmjerena razlika između nazivnoga i stvarnoga otvora dovoljna
da objasni zadani protok. Sintetičke podatke koristimo za provjeru modela.


In [ ]:
# Z4: tlakovi u bar, zatim omjeri apsolutnih tlakova.
p_atm, p0_gauge = 1.0, 7.0
pb_gauge = np.array([4.0, 3.0])
p0_abs = p0_gauge + p_atm
pb_abs = pb_gauge + p_atm
p_star = p0_abs*critical_pressure_ratio()
pe = np.maximum(pb_abs, p_star)
Me = np.sqrt(2/(gamma-1)*((p0_abs/pe)**((gamma-1)/gamma)-1))
assert np.allclose(pe, [5.0, 4.2262543], atol=1e-7, rtol=0)
assert np.allclose(Me, [0.84770495, 1.0], atol=1e-7, rtol=0)
print("Z4: pb(abs) / pe(abs) [bar] / Ma_e")
for pb, pressure, mach in zip(pb_abs, pe, Me):
    print(f"    {pb:.3f} / {pressure:.6f} / {mach:.6f}")

# Z5: neovisno izmjerena stvarna površina, koeficijent nije unaprijed zadan.
p0_z5, T0_z5, pb_z5 = 600e3, 300.0, 100e3
Ag, mdot_measured = 48e-6, 0.0595
ideal_z5 = choked_mass_flow(p0_z5, T0_z5, Ag)
Cd_estimated = mdot_measured/ideal_z5
CdAg = mdot_measured/choked_mass_flow(p0_z5, T0_z5, 1.0)
assert pb_z5/p0_z5 < critical_pressure_ratio()
assert np.isclose(ideal_z5, 0.0672064865, atol=1e-10, rtol=0)
assert np.isclose(Cd_estimated, 0.885331209, atol=1e-9, rtol=0)
assert np.isclose(CdAg*1e6, 42.49589804, atol=1e-8, rtol=0)
assert ideal_z5 > mdot_measured
print(f"Z5: idealno {ideal_z5:.7f} kg/s; CdAg = {CdAg*1e6:.5f} mm²; Cd = {Cd_estimated:.6f}")


## Protumači

Plato nije numeričko zasićenje nego promjena fizikalnog ograničenja. Za validaciju koeficijenta istjecanja treba mjeriti stvarni maseni protok i stagnacijske veličine te iskazati njihovu nesigurnost; slaganje jedne radne točke nije dokaz valjanosti u cijelom rasponu.


1. Zašto nestlačiva procjena ne prikazuje plato, iako se slaže s izentropskom
   procjenom pri malom padu tlaka?
2. U kojem slučaju Z4 daljnje snižavanje protutlaka mijenja izlazni Machov broj,
   a u kojem se prilagodba odvija izvan sapnice?
3. Što bi ostalo nepoznato u Z5 da nema neovisnog mjerenja slobodnog otvora?
